# Lab 18.2: Build a Steering Vector, and Check It Like an Explanation

Chapter 18, Steering, Editing and Monitoring Models. Run the cells in order; the only cell you may edit is marked **Your turn**. GPT-2 small (124M parameters) runs on a laptop CPU through TransformerLens. The first run downloads the model (about 550 MB) from the Hugging Face Hub.

The plan: build a contrastive (difference-of-means) steering vector for positive versus negative sentiment from short finance sentences, add it to the residual stream, and test it the way Chapter 18 asks: a dose-response curve, a random vector of the same size as a control, a side-effect measure on neutral text, a layer sweep, and a reading test with the same vector.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch
from transformer_lens import HookedTransformer
torch.manual_seed(0)
model = HookedTransformer.from_pretrained("gpt2", device="cpu")
model.eval()
print("layers:", model.cfg.n_layers, "width:", model.cfg.d_model)

Loaded pretrained model gpt2 into HookedTransformer
layers: 12 width: 768


In [2]:
NOUNS = ["outlook for the fund", "quarterly report", "new savings account", "service at the branch",
         "loan offer", "advisor's forecast", "bank's earnings call", "mortgage process",
         "market today", "insurance claim experience"]
POS = ["excellent", "wonderful", "fantastic", "delightful", "superb"]
NEG = ["terrible", "awful", "horrible", "dreadful", "disastrous"]
PAIRS_P = [f"The {n} was {w}." for n in NOUNS for w in POS]
PAIRS_N = [f"The {n} was {w}." for n in NOUNS for w in NEG]
TESTS = ["The quarterly report from the bank was", "Analysts said the new loan product is",
         "My visit to the branch today was", "The customer service at the insurer is",
         "Overall, the fund's performance this year has been", "The new credit card app is",
         "I think the mortgage rates right now are", "The meeting with our financial advisor was"]
PW = [" good", " great", " strong", " positive", " nice"]   # evaluation words, not used to build the vector
NW = [" bad", " poor", " weak", " negative", " awful"]
PID = [model.to_single_token(w) for w in PW]; NID = [model.to_single_token(w) for w in NW]
NEUTRAL = ("The central bank publishes its interest rate decision eight times a year. "
           "A deposit account pays interest on the balance held. Loans are repaid in monthly "
           "installments over an agreed term. The report lists assets, liabilities and equity "
           "at the end of the period.")
NAMES = [f"blocks.{l}.hook_resid_pre" for l in range(12)]

def acts(texts):
    """Residual stream before each layer, at the sentiment word (the token before the period)."""
    out = []
    with torch.no_grad():
        for t in texts:
            _, c = model.run_with_cache(model.to_tokens(t), names_filter=lambda n: n in NAMES)
            out.append(np.stack([c[n][0, -2].numpy() for n in NAMES]))
    return np.array(out)

AP, AN = acts(PAIRS_P), acts(PAIRS_N)
print("activations:", AP.shape)

activations: (50, 12, 768)


## Step 1: the vector, and is it stable?

The steering vector at layer 6 is the mean activation on positive sentences minus the mean on negative ones. Before trusting it, build it twice from two disjoint halves of the nouns and compare the two directions. **Checkpoint 1:** the vector's length is about 37 and the two half-vectors point in almost the same direction (cosine above 0.9).

In [3]:
LAYER = 6
v = AP[:, LAYER].mean(0) - AN[:, LAYER].mean(0)
half = np.array([i // 5 < 5 for i in range(50)])
v1 = AP[half, LAYER].mean(0) - AN[half, LAYER].mean(0)
v2 = AP[~half, LAYER].mean(0) - AN[~half, LAYER].mean(0)
cos = v1 @ v2 / np.linalg.norm(v1) / np.linalg.norm(v2)
print(f"length of v: {np.linalg.norm(v):.1f}; split-half cosine: {cos:.3f}")
rng = np.random.default_rng(0)
def rand_like(vec, seed):
    r = np.random.default_rng(seed).standard_normal(vec.shape)
    return r / np.linalg.norm(r) * np.linalg.norm(vec)

length of v: 36.8; split-half cosine: 0.995


In [4]:
def run(layer, vec, alpha, text, mode):
    vt = torch.tensor(vec, dtype=torch.float32)
    hook = lambda x, hook: x + alpha * vt          # add the vector at every position
    toks = model.to_tokens(text)
    with torch.no_grad():
        logits = model.run_with_hooks(toks, fwd_hooks=[(f"blocks.{layer}.hook_resid_pre", hook)])
    if mode == "sent":                             # positive minus negative next-word logit
        l = logits[0, -1]
        return (l[PID].mean() - l[NID].mean()).item()
    lp = torch.log_softmax(logits[0, :-1], -1)     # mean loss per token on a neutral text
    return -lp.gather(1, toks[0, 1:, None]).mean().item()

## Step 2: dose-response with a random control

**Checkpoint 2:** with no steering the mean logit gap is about +1.6. It rises with positive strength and falls below zero with negative strength; a random vector of the same length barely moves it. The loss on the neutral finance text rises as you push harder in either direction, for the random vector too.

In [5]:
alphas = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
rs = [rand_like(v, s) for s in range(3)]
print(" alpha  steer  random   loss(steer) loss(random)  min..max over prompts")
for a in alphas:
    s = [run(LAYER, v, a, t, "sent") for t in TESTS]
    r = np.mean([[run(LAYER, rv, a, t, "sent") for t in TESTS] for rv in rs])
    ce = run(LAYER, v, a, NEUTRAL, "ce"); cer = np.mean([run(LAYER, rv, a, NEUTRAL, "ce") for rv in rs])
    print(f"{a:+5.1f} {np.mean(s):+6.2f} {r:+7.2f} {ce:10.2f} {cer:11.2f}     {min(s):+.2f}..{max(s):+.2f}")

 alpha  steer  random   loss(steer) loss(random)  min..max over prompts


 -1.5  -2.21   +0.74       5.28        4.36     -3.01..-1.73


 -1.0  -2.10   +0.81       4.49        3.78     -3.37..-1.37


 -0.5  -1.37   +1.15       3.71        3.43     -2.92..-0.05


 +0.0  +1.64   +1.64       3.33        3.33     +0.16..+3.38


 +0.5  +4.62   +1.98       3.45        3.47     +2.58..+5.84


 +1.0  +6.24   +2.22       3.91        3.98     +4.24..+7.20


 +1.5  +7.00   +2.46       4.74        4.90     +5.54..+7.50


 +2.0  +7.17   +2.50       5.56        6.03     +6.28..+7.85


## Step 3: read the text, not only the numbers

Greedy continuations of one prompt at several strengths. **Checkpoint 3:** at strength 1 the continuation turns approving; at strength 2 the text degrades into repetition. Numbers alone would not show you that the "more positive" model at strength 2 is also a broken one.

In [6]:
def generate(prompt, alpha, n=20):
    vt = torch.tensor(v, dtype=torch.float32)
    hook = lambda x, hook: x + alpha * vt
    with model.hooks(fwd_hooks=[(f"blocks.{LAYER}.hook_resid_pre", hook)]):
        out = model.generate(model.to_tokens(prompt), max_new_tokens=n, do_sample=False, verbose=False)
    return model.to_string(out[0][1:])
for a in [-1.0, 0.0, 1.0, 2.0]:
    print(f"strength {a:+.1f}: {generate('The quarterly report from the bank was', a)!r}")

strength -1.0: 'The quarterly report from the bank was released on Thursday, and the government has been forced to pay billions of dollars in loans to banks and'


strength +0.0: "The quarterly report from the bank was released on Tuesday, and it shows that the bank's balance sheet has grown by $1.3"


strength +1.0: 'The quarterly report from the bank was released by the Bank of America Merrill Lynch.\n\nThe report is a great way to get a'


strength +2.0: 'The quarterly report from the bank was presented to the company of the paper. The paper is also presented to the paper of the paper.'


## Step 4: where in the network?

Build the vector at each layer and add it at strength 0.5 (each layer's own vector, so the push is in proportion to that layer's activations). **Checkpoint 4:** early layers give small effects; the middle and late layers give the largest; the random vector stays near zero everywhere while still costing fluency.

In [7]:
base = np.mean([run(0, np.zeros(768), 0.0, t, "sent") for t in TESTS])
ce0 = run(0, np.zeros(768), 0.0, NEUTRAL, "ce")
print("layer  effect  random   extra loss (steer, random)")
for l in range(12):
    vl = AP[:, l].mean(0) - AN[:, l].mean(0); rl = rand_like(vl, 0)
    e = np.mean([run(l, vl, 0.5, t, "sent") for t in TESTS]) - base
    er = np.mean([run(l, rl, 0.5, t, "sent") for t in TESTS]) - base
    print(f"{l:5d} {e:+7.2f} {er:+7.2f}   {run(l, vl, 0.5, NEUTRAL, 'ce') - ce0:+.2f} {run(l, rl, 0.5, NEUTRAL, 'ce') - ce0:+.2f}")

layer  effect  random   extra loss (steer, random)


    0   +1.02   -0.01   +0.09 +0.16


    1   +1.28   +0.16   +0.14 +0.05


    2   +1.69   +0.21   +0.16 +0.08


    3   +1.63   +0.16   +0.07 +0.09


    4   +2.12   +0.05   +0.12 +0.19


... (3 outputs left out)

    8   +3.05   -0.10   +0.14 +0.20


    9   +3.39   +0.03   +0.08 +0.15


   10   +2.46   +0.20   +0.03 +0.10


   11   +1.59   +0.12   +0.02 +0.11


## Step 5: the same vector as a monitor

Project held-out sentences (new nouns, new sentiment words) on the layer-6 vector, centered at the midpoint of the two training means. A positive score reads as positive. **Checkpoint 5:** the held-out AUROC is close to 1, but most negated negatives ("not terrible") are read as negative, although a person would read them as mildly positive. A reading vector inherits the shortcuts of the sentences that built it.

In [8]:
HN = ["trip to the bank", "pension statement", "card upgrade", "call with the insurer", "investment seminar",
      "tax refund process", "branch renovation", "budgeting app", "fee schedule", "claims hotline"]
u = v / np.linalg.norm(v)
mid = (AP[:, LAYER].mean(0) + AN[:, LAYER].mean(0)) / 2 @ u
score = lambda words: acts([f"The {n} was {w}." for n in HN for w in words])[:, LAYER] @ u - mid
sp = score(["great", "lovely", "brilliant", "pleasant", "impressive"])
sn = score(["bad", "poor", "miserable", "unpleasant", "disappointing"])
sx = score(["not terrible", "not awful", "not bad", "not disappointing", "not horrible"])
auroc = ((sp[:, None] > sn[None, :]).mean())
print(f"held-out AUROC {auroc:.3f}; positives read positive {np.mean(sp > 0):.2f}; "
      f"negatives read negative {np.mean(sn < 0):.2f}; negated negatives read negative {np.mean(sx < 0):.2f}")

held-out AUROC 1.000; positives read positive 1.00; negatives read negative 1.00; negated negatives read negative 0.78


## Your turn

Replace the two word lists below with a contrast you care about, for example cautious versus promotional language about an investment ("may", "could", "historically" against "guaranteed", "certain", "risk-free"), rebuild the vector and repeat Steps 2 and 5. Record the split-half cosine, the dose-response with its random control, the side effect at strength 1, and one failure you found by reading generations.

In [9]:
MY_POS = ["excellent", "wonderful", "fantastic", "delightful", "superb"]   # edit me
MY_NEG = ["terrible", "awful", "horrible", "dreadful", "disastrous"]       # edit me
MP = acts([f"The {n} was {w}." for n in NOUNS for w in MY_POS])
MN = acts([f"The {n} was {w}." for n in NOUNS for w in MY_NEG])
mv = MP[:, LAYER].mean(0) - MN[:, LAYER].mean(0)
print("cosine with the sentiment vector:", round(float(mv @ v / np.linalg.norm(mv) / np.linalg.norm(v)), 3))
for a in [0.0, 1.0]:
    print(f"strength {a}: {generate('The new investment fund is', a)!r}")

cosine with the sentiment vector: 1.0


strength 0.0: 'The new investment fund is expected to be worth $1.5 billion, according to a report by the Financial Times.\n'


strength 1.0: 'The new investment fund is a great way to get your portfolio to grow and to share your portfolio with others.\n\nThe'
